# Full Pipeline Integration — Strict MATLAB-Alignment

This notebook runs the strict `separation` pipeline and validates each stage against
strict fixtures generated from the same pipeline contract.

**Stage chain:** neural_enhancement → movement_correction → seeds-cleansed source extraction → component filtering/background update → deconvolution

## 1. Setup & Imports

In [1]:

import sys, pickle
from pathlib import Path

SEP_DIR = Path(".").resolve()
# walk up until we find the separation folder
while SEP_DIR.name != "separation" and SEP_DIR != SEP_DIR.parent:
    SEP_DIR = SEP_DIR.parent
if SEP_DIR.name != "separation":
    SEP_DIR = Path(".").resolve()

sys.path.insert(0, str(SEP_DIR))
REPO_ROOT = SEP_DIR.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))
TEST_DATA = SEP_DIR / "_test_data"
TEST_DATA_STRICT = SEP_DIR / "_test_data_strict"
print(f"Separation dir: {SEP_DIR}")
print(f"Repo root: {REPO_ROOT}")


Separation dir: /home/yz/MIN1PIPE/separation
Repo root: /home/yz/MIN1PIPE


In [2]:

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# ── Inline visualization helpers (from MIN1PIPE repo) ──

def normalize_vis(frame_in, dim=None):
    """Normalize intensity to [0, 1]. (from utilities/elements/normalize.m)"""
    arr = np.asarray(frame_in, dtype=np.float64)
    if arr.size == 0:
        return arr
    if dim is None or int(dim) == 4:
        mn = np.nanmin(arr)
        out = arr - mn
        mx = np.nanmax(out)
        if mx == 0 or np.isnan(mx):
            return np.zeros_like(out)
        return out / mx
    axis = int(dim) - 1 if int(dim) > 0 else int(dim)
    mn = np.nanmin(arr, axis=axis, keepdims=True)
    out = arr - mn
    mx = np.nanmax(out, axis=axis, keepdims=True)
    safe = np.where(mx == 0, 1.0, mx)
    out = out / safe
    return np.where(mx == 0, 0.0, out)


def plot_contour_standalone(roifn, sigfn, seedsfn, imax, pixh, pixw, ax=None):
    """Plot ROI contours on max-projection. (from postprocess/plot_contour.m)"""
    roi = np.asarray(roifn, dtype=np.float64)
    sig = np.asarray(sigfn, dtype=np.float64)
    seeds = np.asarray(seedsfn).reshape(-1)
    imax_arr = np.asarray(imax, dtype=np.float64)
    pixh_i, pixw_i = int(pixh), int(pixw)
    n_pixels = pixh_i * pixw_i

    if roi.shape[0] != n_pixels and roi.shape[1] == n_pixels:
        roi = roi.T

    if ax is None:
        ax = plt.gca()
    ax.imshow(imax_arr, vmin=0.0, vmax=0.8, cmap="viridis", origin="upper",
              interpolation="nearest")

    n_ids = min(roi.shape[1], sig.shape[0], seeds.shape[0])
    for idx in range(n_ids):
        tmp = roi[:, idx].reshape((pixh_i, pixw_i)) * float(np.max(sig[idx, :]))
        tmp = gaussian_filter(tmp, sigma=3.0)
        level = float(np.max(tmp) * 0.8)
        if not np.isfinite(level) or level <= 0:
            continue
        cs = ax.contour(np.flipud(tmp), levels=[level], colors="none")
        for seg in cs.allsegs[0]:
            if seg.shape[0] < 2:
                continue
            ax.plot(seg[:, 0], pixh_i - seg[:, 1], "r", linewidth=1.0)
        if hasattr(cs, "remove"):
            cs.remove()
        y, x = np.unravel_index(int(seeds[idx]), (pixh_i, pixw_i))
        ax.text(x + 1, y + 1, str(idx + 1), fontsize=9, color="white")

    ax.set_title("Neural Contours")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0, pixw_i - 1)
    ax.set_ylim(pixh_i - 1, 0)


def plot_traces(sigfn, ax=None, title="Traces"):
    """Plot stacked normalized traces. (from demo_min1pipe.m)"""
    if ax is None:
        ax = plt.gca()
    sigt = np.asarray(sigfn, dtype=np.float64).copy()
    for i in range(sigt.shape[0]):
        sigt[i, :] = normalize_vis(sigt[i, :])
    ax.plot((sigt + np.arange(1, sigt.shape[0] + 1)[:, None]).T)
    ax.axis("tight")
    ax.set_title(title)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Component")


def plot_mc_scores(raw_score, corr_score, ax=None, title="MC Scores"):
    """Plot motion correction quality scores."""
    if ax is None:
        ax = plt.gca()
    ax.plot(raw_score, label="raw_score", alpha=0.8)
    ax.plot(corr_score, label="corr_score", alpha=0.8)
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Displacement (px)")


def similarity_report(name, actual, expected, rtol=1e-5, atol=1e-7):
    """Compute and print similarity metrics between two arrays."""
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    if actual.shape != expected.shape:
        print(f"  {name}: SHAPE MISMATCH actual={actual.shape} expected={expected.shape}")
        return False
    abs_diff = np.abs(actual - expected)
    max_abs = float(np.max(abs_diff))
    mean_abs = float(np.mean(abs_diff))
    exp_max = float(np.max(np.abs(expected)))
    rel_err = max_abs / max(exp_max, 1e-10)
    corr = float(np.corrcoef(actual.ravel(), expected.ravel())[0, 1]) if actual.size > 1 else 1.0
    match = np.allclose(actual, expected, rtol=rtol, atol=atol)
    status = "PASS" if match else "FAIL"
    print(f"  [{status}] {name}:")
    print(f"       shape={actual.shape}  max_abs_diff={max_abs:.2e}  "
          f"mean_abs_diff={mean_abs:.2e}  rel_err={rel_err:.2e}  corr={corr:.6f}")
    return match


def plot_comparison_images(img1, img2, title1, title2, suptitle=""):
    """Side-by-side image comparison with difference map."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    if suptitle:
        fig.suptitle(suptitle, fontsize=14)
    axes[0].imshow(img1, cmap="viridis", origin="upper", interpolation="nearest")
    axes[0].set_title(title1)
    axes[1].imshow(img2, cmap="viridis", origin="upper", interpolation="nearest")
    axes[1].set_title(title2)
    diff = np.abs(np.asarray(img1, dtype=np.float64) - np.asarray(img2, dtype=np.float64))
    im = axes[2].imshow(diff, cmap="hot", origin="upper", interpolation="nearest")
    axes[2].set_title(f"Abs Difference (max={float(np.max(diff)):.2e})")
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    plt.tight_layout()
    plt.show()


In [3]:
from _shared.params import strict_default_parameters
from _shared.fixtures import load_npz
from pipeline_strict import run_full_pipeline_strict

video_path = REPO_ROOT / "demo" / "demo_data.tif"
strict_dir = TEST_DATA_STRICT
print(f"Video path: {video_path}")
print(f"Strict fixture root: {strict_dir}")

Video path: /home/yz/MIN1PIPE/demo/demo_data.tif
Strict fixture root: /home/yz/MIN1PIPE/separation/_test_data_strict


## 2. Load Strict Fixture References

In [4]:
refs = {}
for mod in ["motion_correction", "source_detection", "component_filtering", "calcium_deconvolution"]:
    p = strict_dir / mod / "output.npz"
    if not p.exists():
        raise FileNotFoundError(f"Missing strict fixture: {p}. Run `python separation/build_strict_fixtures.py` first.")
    refs[mod] = load_npz(p)
print("Loaded strict fixtures for all stages.")

Loaded strict fixtures for all stages.


## 3. Run Strict Pipeline

In [5]:
params = strict_default_parameters()
result = run_full_pipeline_strict(video_path=video_path, params=params)
mc_result = result.motion
sd_result = result.source
cf_result = result.component
cd_result = result.deconv
print(f"Motion corrected video: {mc_result.corrected_video.shape}")
print(f"Detected components: {sd_result.n_components}")
print(f"Filtered ROI/signal: {cf_result.roifn.shape} / {cf_result.sigfn.shape}")
print(f"Deconv outputs: spk={cd_result.spkfn.shape}, dff={cd_result.dff.shape}")

Motion corrected video: (1000, 150, 150)
Detected components: 15
Filtered ROI/signal: (22500, 15) / (15, 1000)
Deconv outputs: spk=(15, 1000), dff=(15, 1000)


## 4. Stage-by-Stage Numeric Validation

In [6]:
stage_results = {}

print("\n[Stage 1] motion_correction")
s1 = True
s1 &= similarity_report("corrected_video", mc_result.corrected_video, refs["motion_correction"]["corrected_video"], rtol=1e-4, atol=1e-6)
s1 &= similarity_report("imax", mc_result.imax, refs["motion_correction"]["imax"], rtol=1e-4, atol=1e-6)
s1 &= similarity_report("raw_score", mc_result.raw_score, refs["motion_correction"]["raw_score"], rtol=1e-4, atol=1e-6)
s1 &= similarity_report("corr_score", mc_result.corr_score, refs["motion_correction"]["corr_score"], rtol=1e-4, atol=1e-6)
stage_results["motion_correction"] = s1

print("\n[Stage 2] source_detection")
s2 = True
s2 &= similarity_report("roifn", sd_result.roifn, refs["source_detection"]["roifn"], rtol=1e-4, atol=1e-6)
s2 &= similarity_report("sigfn", sd_result.sigfn, refs["source_detection"]["sigfn"], rtol=1e-4, atol=1e-6)
s2 &= similarity_report("seedsfn", sd_result.seedsfn, refs["source_detection"]["seedsfn"], rtol=1e-5, atol=1e-7)
stage_results["source_detection"] = s2

print("\n[Stage 3] component_filtering")
s3 = True
s3 &= similarity_report("roifn", cf_result.roifn, refs["component_filtering"]["roifn"], rtol=1e-4, atol=1e-6)
s3 &= similarity_report("sigfn", cf_result.sigfn, refs["component_filtering"]["sigfn"], rtol=1e-4, atol=1e-6)
s3 &= similarity_report("bgfn", cf_result.bgfn, refs["component_filtering"]["bgfn"], rtol=1e-4, atol=1e-6)
s3 &= similarity_report("bgffn", cf_result.bgffn, refs["component_filtering"]["bgffn"], rtol=1e-4, atol=1e-6)
stage_results["component_filtering"] = s3

print("\n[Stage 4] calcium_deconvolution")
s4 = True
s4 &= similarity_report("spkfn", cd_result.spkfn, refs["calcium_deconvolution"]["spkfn"], rtol=1e-4, atol=1e-6)
s4 &= similarity_report("dff", cd_result.dff, refs["calcium_deconvolution"]["dff"], rtol=1e-4, atol=1e-6)
stage_results["calcium_deconvolution"] = s4


[Stage 1] motion_correction


  [PASS] corrected_video:
       shape=(1000, 150, 150)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_err=0.00e+00  corr=1.000000
  [PASS] imax:
       shape=(150, 150)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_err=0.00e+00  corr=1.000000
  [PASS] raw_score:
       shape=(1000,)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_err=0.00e+00  corr=nan
  [PASS] corr_score:
       shape=(1000,)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_err=0.00e+00  corr=nan

[Stage 2] source_detection
  [PASS] roifn:
       shape=(22500, 15)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_err=0.00e+00  corr=1.000000
  [PASS] sigfn:
       shape=(15, 1000)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_err=0.00e+00  corr=1.000000
  [PASS] seedsfn:
       shape=(15,)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_err=0.00e+00  corr=1.000000

[Stage 3] component_filtering
  [PASS] roifn:
       shape=(22500, 15)  max_abs_diff=0.00e+00  mean_abs_diff=0.00e+00  rel_er

/home/yz/MIN1PIPE/.venv-min1pipe/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


## 5. MATLAB-Style 2x3 Panel

In [7]:
fig = plt.figure(figsize=(12.8, 9.0))

ax1 = fig.add_subplot(2, 3, 1)
ax1.imshow(mc_result.imaxn, cmap="viridis", origin="upper", interpolation="nearest")
ax1.set_title("Raw (demo_min1pipe.m)")
ax1.set_box_aspect(1)

ax2 = fig.add_subplot(2, 3, 2)
ax2.imshow(mc_result.imaxy_pre, cmap="viridis", origin="upper", interpolation="nearest")
ax2.set_title("Before MC (demo_min1pipe.m)")
ax2.set_box_aspect(1)

ax3 = fig.add_subplot(2, 3, 3)
ax3.imshow(mc_result.imax, cmap="viridis", origin="upper", interpolation="nearest")
ax3.set_title("After MC (demo_min1pipe.m)")
ax3.set_box_aspect(1)

ax4 = fig.add_subplot(2, 3, 4)
plot_contour_standalone(cf_result.roifn, cf_result.sigfn, cf_result.seedsfn, mc_result.imax, mc_result.pixh, mc_result.pixw, ax=ax4)
ax4.set_title("Neural Contours (plot_contour.m)")
ax4.set_box_aspect(1)

ax5 = fig.add_subplot(2, 3, 5)
plot_mc_scores(mc_result.raw_score, mc_result.corr_score, ax=ax5, title="MC Scores (demo_min1pipe.m)")
ax5.set_box_aspect(1)

ax6 = fig.add_subplot(2, 3, 6)
plot_traces(cf_result.sigfn, ax=ax6, title="Traces (demo_min1pipe.m)")
ax6.set_box_aspect(1)

fig.suptitle("MIN1PIPE Strict Separation Output", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 6. Optional MATLAB Golden Comparison

In [8]:
matlab_ref = REPO_ROOT / "artifacts" / "golden" / "matlab" / "demo_data" / "latest" / "demo_data_data_processed.mat"
if matlab_ref.exists():
    from min1pipe.io import load_processed_mat
    mat = load_processed_mat(matlab_ref, source_layout="matlab")
    print("MATLAB golden artifact found.")
    print(f"  MATLAB imax shape: {mat['imax'].shape}")
    print(f"  Python imax shape: {mc_result.imax.shape}")
    if mat["imax"].shape == mc_result.imax.shape:
        similarity_report("imax vs MATLAB", mc_result.imax, mat["imax"], rtol=5e-2, atol=1e-2)
    else:
        print("  Shape mismatch is expected if params differ.")
else:
    print("MATLAB golden artifact not found; skipping cross-check.")

MATLAB golden artifact found.
  MATLAB imax shape: (150, 150)
  Python imax shape: (150, 150)
  [FAIL] imax vs MATLAB:
       shape=(150, 150)  max_abs_diff=8.17e-01  mean_abs_diff=2.66e-02  rel_err=1.10e+00  corr=0.831568


## 7. Summary

In [9]:
all_pass = True
for stage, passed in stage_results.items():
    print(f"{stage:22s}: {'PASS' if passed else 'FAIL'}")
    all_pass = all_pass and passed
print("\nOverall strict fixture parity:", "PASS" if all_pass else "FAIL")

motion_correction     : PASS
source_detection      : PASS
component_filtering   : PASS
calcium_deconvolution : PASS

Overall strict fixture parity: PASS
